# 02. Feature Engineering: Building the AI Brain

In this notebook, we transform raw hourly demand data into a high-signal feature set for our forecasting models. 

### Key Steps:
1. **Data Acquisition**: Fetching aggregated data from BigQuery.
2. **Memory Optimization**: Downcasting types to save RAM.
3. **Cyclical Features**: Using Sine/Cosine for time continuity.
4. **Lag Features**: Creating historical memory (Short-term and Seasonal lags).
5. **Feature Validation**: Visualizing the new feature space.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from dotenv import load_dotenv
import sys

# Add src to path to use our pipeline modules
sys.path.append('..')
from src.features.build_features import create_lag_features, create_temporal_features

load_dotenv('../.env')
client = bigquery.Client()

## 1. Load Cleaned Data from BigQuery
We join `Fact_Demand_Hourly` with `Dim_Time` to get rich temporal attributes like `is_rush_hour` and `day_of_week_number`.

In [ ]:
project_id = os.getenv("BQ_PROJECT_ID")
dataset_id = os.getenv("BQ_DATASET_ID", "nyc_taxi_dw")

query = f"""
SELECT 
    f.pickup_time_key, 
    f.pulocation_id, 
    f.service_type_key,
    f.total_demand,
    t.hour,
    t.day_of_week_number,
    t.is_rush_hour,
    t.is_weekend
FROM `{project_id}.{dataset_id}.Fact_Demand_Hourly` f
JOIN `{project_id}.{dataset_id}.Dim_Time` t ON f.pickup_time_key = t.Time_Key
ORDER BY f.pulocation_id, f.pickup_time_key
"""

print("Fetching data from BigQuery...")
df = client.query(query).to_dataframe()
print(f"Loaded {len(df)} rows.")

## 2. RAM Optimization (Downcasting)
This step is critical for handling large datasets on consumer hardware.

In [ ]:
def optimize_memory(df):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory reduced from {start_mem:.2f} MB to {end_mem:.2f} MB')
    return df

df = optimize_memory(df)

## 3. Cyclical Time Features
Transforming `hour` (0-23) into `sin` and `cos` coordinates so the model understands that hour 23 is close to hour 0.

In [ ]:
df = create_temporal_features(df)

# Visualization of Cyclical Encoding
sample_hours = df[['hour', 'hour_sin', 'hour_cos']].drop_duplicates().sort_values('hour')
plt.figure(figsize=(6,6))
plt.scatter(sample_hours['hour_sin'], sample_hours['hour_cos'])
for i, row in sample_hours.iterrows():
    plt.annotate(int(row['hour']), (row['hour_sin'], row['hour_cos']))
plt.title("Cyclical Encoding of Hours")
plt.xlabel("Sine")
plt.ylabel("Cosine")
plt.grid(True)
plt.show()

## 4. Lag Features (Model Memory)
We create features based on what happened 1 hour ago, 24 hours ago (yesterday), and 168 hours ago (last week).

In [ ]:
df_final = create_lag_features(df, lags=[1, 2, 24, 168])

print("Features created:")
print(df_final.columns.tolist())
df_final.head()

## 5. Feature Analysis: Correlation Heatmap
Checking which historical lags have the strongest relationship with current demand.

In [ ]:
plt.figure(figsize=(10, 8))
corr = df_final[['total_demand', 'lag_demand_1h', 'lag_demand_24h', 'lag_demand_168h', 'hour_sin', 'hour_cos']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Feature Correlation Matrix")
plt.show()

## 6. Conclusion
We now have a robust feature set ready for training. 
- **Lags** capture the immediate and seasonal trends.
- **Cyclical features** provide smooth temporal context.
- **Downcasting** ensures we can train models efficiently.

Proceed to `03_train_xgboost_model.ipynb` for the training phase.